In my own computations (by hand), I forgot to use the inner product that satisfies the Morimoto criterion, hence the $c_{ijk}$ in addition to the $f_{ijk}$. When restricted to $V=\text{span}_\mathbb{R}(\varepsilon_1,\ldots,\varepsilon_{2m})$, the $\varepsilon_i$ for an orthonormal basis with

$$ |\varepsilon_i|^2 = \frac{(i-1)!}{(2m-i)!}\quad \text{hence}\quad |\varepsilon_i^*\wedge \varepsilon_j^*\otimes \varepsilon_k|^2 = \frac{(2m-i)!}{(i-1)!}\frac{(2m-j)!}{(j-1)!}\frac{(k-1)!}{(2m-k)!}$$

This notebook will compute the conditions under which $f^{ijk}\varepsilon_{i}^*\otimes\varepsilon_{j}^*\otimes\varepsilon_{k}$ is in $\text{Im}(S^1)^\perp$, where $S^1:V^*\otimes\mathfrak{gl}(2,\mathbb{R})\to V^* \wedge V^*\otimes V$ is the first spencer operator, defined by

$$S^1\phi(v_1,v_2)=\phi(v_1)(v_2)-\phi(v_2)(v_1)$$

Then, these conditions will be used to produce a basis for $\text{Im}(S^1)^\perp$.

Remark: If I wanted to make this notebook more time efficient, one way I could do that is by replacing any large coefficient dictionaries and using the attribute .as_coefficient_dict for add objects...A new function substitute_f_for_c would not be so hard to implement

In [1]:
from sympy import *
import time
from collections import defaultdict
import copy
import pickle

In [2]:
# We'll work with gl2+heis(2*m+1)
m=6

In [3]:
# i=Idx('i',range=(1,2*m+1))
# j=Idx('j',range=(1,2*m+1))
# k=Idx('k',range=(1,2*m+1))
c=IndexedBase('c',shape=(2*m,2*m,2*m))
f=IndexedBase('f',shape=(2*m,2*m,2*m))
e=IndexedBase('e',shape=(2*m,2*m,2*m))
Y,H,E,X=symbols('Y,H,E,X')
C2_basis=[e[a,b,c] for a in range(1,2*m+1) for b in range(a+1,2*m+1) for c in range(1,2*m+1)]

# Firstly, set the antisymmetric relations
antisymm_subs={}
for a in range(1,2*m+1):
    for d in range(1,2*m+1):
        antisymm_subs[c[a,a,d]]=0
        antisymm_subs[f[a,a,d]]=0
        antisymm_subs[e[a,a,d]]=0
        for b in range(a+1,2*m+1):
            antisymm_subs[c[b,a,d]]=-c[a,b,d]
            antisymm_subs[f[b,a,d]]=-f[a,b,d]
            antisymm_subs[e[b,a,d]]=-e[a,b,d]


## Helper Methods

In [4]:
def find_basis(SpanningSet,Basis):
    '''args: SpanningSet, a list of LinCombs, and Basis, a list of symbols
       Returns: A list of LinCombs which are a basis for the subspace given by SpanningSet'''
    ThisMat=zeros(len(SpanningSet),len(Basis))
    for i in range(len(SpanningSet)):
        set_row(ThisMat,i,coordinatize(Basis,SpanningSet[i]))
    ThisMat=ThisMat.rref()[0]
    Result=[list(ThisMat.row(i)) for i in range(shape(ThisMat)[0]) if list(ThisMat.row(i))!=[0]*len(ThisMat.row(i))]
    return([coords_to_lin_comb(A,Basis) for A in Result])

In [5]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    if type(col)==type(zeros(2,2)):
        colList=list(col)
    else:
        if type(col)==type([0]):
            colList=col
        else: print('SetCol error: arg col must be either matrix or list')
            
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [6]:
def coordinatize(basis,linearComb):
    '''args: a list of symbols and a LinComb of those symbols
       Returns: Vector representation of linearComb w.r.t basis'''
    dim=len(basis)
    # Deal with the zero case
    if linearComb==0:
        return [0]*dim
    
    # Check for symbols not in the basis:
    expr=linearComb
    for A in basis:
        expr=expr.subs(A,0)
    if expr!=0:
        print('coordinatize error: |linearComb| has symbol',expr, 'not from |basis|')
        return None
    
    exprList=[0]*dim
    exprList[0]=linearComb
    coords=[0]*dim
    
    for i in range(1,dim):
        exprList[i]=exprList[i-1].subs(basis[i-1],0)
    
    lastCoord=exprList[dim-1].subs(basis[dim-1],1)
    coords[dim-1]=lastCoord
    
    for j in range(dim-1):
        coords[j]=(exprList[j]-exprList[j+1]).subs(basis[j],1)
    
    return coords

def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [7]:
## This uses the 'Zassenhaus Algorithm' to find the interesection of subspaces

def intersection(VecList1,VecList2):
    '''args: lists of vectors of some length; spanning sets for VS V1 and V2
       returns: a basis for the space V1\capV2'''
    if len(VecList1)==0 or len(VecList2)==0:
        return []
    VecLength=len(VecList1[0])
    ThisMat=zeros(len(VecList1)+len(VecList2),2*VecLength)
    for i in range(len(VecList1)):
        set_row(ThisMat,i,VecList1[i]*2)
    for i in range(len(VecList1),len(VecList1)+len(VecList2)):
        set_row(ThisMat,i,VecList2[i-len(VecList1)]+[0]*VecLength)
    IntersectionMat=ThisMat.rref()[0]
    result=[]
    for RowIndex in range(len(IntersectionMat.col(0))):
        ThisRow=list(IntersectionMat.row(RowIndex))
        if ThisRow!=[0]*2*VecLength:
            if ThisRow[0:VecLength]==[0]*VecLength:
                result.append(ThisRow[VecLength:2*VecLength])
    return result

In [8]:
def c_unraveled_index(c1):
    '''c1: An index between 0 and 2*m+1
       returns: the unraveled index of c1 in c'''
    ctr=0
    while c1!=c[ctr//((2*m+1)**2),(ctr//(2*m+1))%(2*m+1),ctr%(2*m+1)]:
        ctr+=1
    return ctr

In [9]:
def f_index_tuple(f_elt):
    '''for argument f[i,j,k], returns (i,j,k)'''
    ctr=0
    i=0
    j=0
    k=0
    while f[i,j,k]!=f_elt:
        ctr+=1
        k=(ctr%(2*m))+1
        j=((ctr//(2*m))%(2*m))+1
        i=(ctr//((2*m)**2))+1
    return (i,j,k)

In [10]:
def update_subs_dicts(subs_dict,gl_elt):
    '''subs_dict: A dict representing a cijk substitution
       result: None
       updates the dicts c_substitutions and total_c_subs, as well
       as determined_cijk'''
    for A in [Y,H,E,X]:
        for key in c_substitutions[A]:
            # 'Back substitute'
            if hasattr(c_substitutions[A][key],'subs'):
                c_substitutions[A][key]=c_substitutions[A][key].subs(subs_dict)
            # Add the substitution to the dict gl_elt
        if A==gl_elt: c_substitutions[A].update(subs_dict)
    for key in total_c_subs:
        if hasattr(total_c_subs[key],'subs'):
            total_c_subs[key]=total_c_subs[key].subs(subs_dict)
    total_c_subs.update(subs_dict)
    for key in subs_dict:
        determined_cijk.add(key)
        t=c_index_tuple(key)
        determined_fijk.add(f[t[0],t[1],t[2]])

In [11]:
def c_index_tuple(c_elt):
    '''for argument c[i,j,k], returns (i,j,k)'''
    ctr=0
    i=0
    j=0
    k=0
    while c[i,j,k]!=c_elt:
        ctr+=1
        k=(ctr%(2*m))+1
        j=((ctr//(2*m))%(2*m))+1
        i=(ctr//((2*m)**2))+1
    return (i,j,k)

In [12]:
def sq_len(i,j,k):
    '''returns |eijk|**2'''
    return (factorial(k-1)*factorial(2*m-i)*factorial(2*m-j)/
            (factorial(2*m-k)*factorial(i-1)*factorial(j-1)))

# Computations

In [13]:
# rels[A][i] is the relation obtained from <f,S^1(ei x A)>=0
rels={Y:[0],H:[0],E:[0],X:[0]}

for i in range(1,2*m+1):
    rels[E].append((sum([c[i,j,j] for j in range(1,2*m+1)])).subs(antisymm_subs))
    rels[X].append((sum([c[i,j,j+1] for j in range(1,2*m)])).subs(antisymm_subs))
    rels[H].append((sum([(2*j-2*m-1)*c[i,j,j] for j in range(1,2*m+1)])).subs(antisymm_subs))
    rels[Y].append((sum([(j-1)*(2*m+1-j)*c[i,j,j-1] for j in range(2,2*m+1)])).subs(antisymm_subs))
    

In [14]:
# This dict substitutes in fijk for cijk with the appropriate coefficients
c_to_f_subs={}
for i in range(1,2*m+1):
    for j in range(i+1,2*m+1):
        for k in range(1,2*m+1):
            c_to_f_subs[c[i,j,k]]=f[i,j,k]*sq_len(i,j,k)

In [15]:
# Let's use the relations to construct substitution dictionaries
# which give the determined_cijk in terms of the free_cijk
# The determined cijk will be the keys of total_c_subs

total_c_subs={}
c_substitutions={'antisymm':{},Y:{},H:{},E:{},X:{}}
f_substitutions={'antisymm':{},Y:{},H:{},E:{},X:{}}
determined_cijk=set()
determined_fijk=set()

# Firstly, set the antisymmetric relations as c_substitutions
for a in range(1,2*m+1):
    for d in range(1,2*m+1):
        c_substitutions['antisymm'][c[a,a,d]]=0
        f_substitutions['antisymm'][f[a,a,d]]=0
        for b in range(a+1,2*m+1):
            c_substitutions['antisymm'][c[b,a,d]]=-c[a,b,d]
            f_substitutions['antisymm'][f[b,a,d]]=-f[a,b,d]
total_c_subs.update(c_substitutions['antisymm'])

for A in [E,X,H,Y]:
    for r in range(1,len(rels[A])):
        relevant_cijk=list(rels[A][r].as_coefficients_dict().keys())
        relevant_cijk.sort(key=c_unraveled_index)
        ctr=0
        while True:
            # Find the least cijk in relevant_cijk which is not yet determined
            # Note that rels[A][r] contains only cijk with i<j

            if not (relevant_cijk[ctr] in total_c_subs):
                new_cijk=relevant_cijk[ctr]
                
                # Add the relation to c_substitutions
                new_subs_dict={new_cijk:solve(rels[A][r].subs(total_c_subs),new_cijk)[0]}
                update_subs_dicts(new_subs_dict,A)
                break
            else: 
                ctr+=1

In [16]:
## Generating a basis for the orthocomplement of S1
free_cijk={c[i,j,k] for i in range(1,2*m+1) for j in range(i+1,2*m+1)
           for k in range(1,2*m+1)}.difference(determined_cijk)

free_fijk={f[i,j,k] for i in range(1,2*m+1) for j in range(i+1,2*m+1)
           for k in range(1,2*m+1)}.difference(determined_fijk)

In [17]:
# Write the remainder of dict f_substitutions using c_substitutions

for A in [E,X,H,Y]:
    for key in c_substitutions[A]:
        t=c_index_tuple(key)
        val=c_substitutions[A][key].subs(c_to_f_subs)/sq_len(t[0],t[1],t[2])
        f_substitutions[A][f[t[0],t[1],t[2]]]=val

# Combine all the subs for f other than antisymm subs
all_f_subs={} # Excluding antisymm subs
for A in [E,X,H,Y]:
    all_f_subs.update(f_substitutions[A])

In [18]:
basis_dict={}

for f_fijk in free_fijk:
    t=f_index_tuple(f_fijk)
    basis_dict[f_fijk]=e[t[0],t[1],t[2]]

for d_fijk in determined_fijk:
    t=f_index_tuple(d_fijk)
    for f_fijk in all_f_subs[d_fijk].as_coefficients_dict():
        coeff=all_f_subs[d_fijk].as_coefficients_dict()[f_fijk]
        basis_dict[f_fijk]+=coeff*e[t[0],t[1],t[2]]

In [19]:
S1_perp_basis=list(basis_dict.values())
S1_perp_vecs=[coordinatize(C2_basis,A) for A in S1_perp_basis]

In [20]:
# # pickle S1_perp_basis 

# file=open('S1_perp_m%d'%m,'wb') # 'wb' means 'write binary mode'
# pickle.dump(S1_perp_basis,file)
# file.close()

## Printing for Computation Checks

In [21]:
# # For checking my computations of the determined cijk
# A=Y
# for key in c_substitutions[A]:
#     print('i =',list(c_substitutions[A]).index(key)+1)
#     display(key
#             ,c_substitutions[A][key])
#     print('\n\n')

In [22]:
# # For checking my computations of the determined fijk
# A=Y
# for key in f_substitutions[A]:
#     print('i =',list(f_substitutions[A]).index(key)+1)
#     display(key
#             ,f_substitutions[A][key])
#     print('\n\n')

In [23]:
# # For checking the basis

# for i in [2]:
#     for k in range(4,2*m+1):
#         for j in [k]:
#             if f[i,j,k] in basis_dict:
#                 print((i,j,k),':')
#                 display(basis_dict[f[i,j,k]])
#                 print('----------------------------------------')

# Checking the Manually Computed Basis

Below is the basis I have computed by hand for the general $m$ case

In [24]:
def delta(a,b):
    if a==b: return 1
    else: return 0

In [25]:
# For checking, use a bunch of sublists like in the manual computation
manual_S1_perp_basis_segmented=[]

manual_S1_perp_basis_segmented.append([e[i,j,1] for i in range(3,2*m+1) 
                               for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,2]-delta(i,3)*e[2,j,1]
                               for i in range(3,2*m+1) for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[1,j,3] 
                               for j in range(5,2*m+1)])
manual_S1_perp_basis_segmented.append([e[2,j,3]-Rational(2*(2*m-2),(2*m-1))*e[1,j,2]
                               for j in range(5,2*m+1)])
manual_S1_perp_basis_segmented.append([e[3,j,3]+e[1,j,1]-2*e[2,j,2]+delta(j,4)*e[2,3,1]
                               for j in range(4,2*m+1)])
manual_S1_perp_basis_segmented.append([e[i,j,3]-delta(i,4)*e[2,j,1]
                               for i in range(4,2*m+1) for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[1,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-2)*2)*e[1,2,3]
                               for k in range(4,2*m+1) for j in range(2,k)])
manual_S1_perp_basis_segmented.append([e[1,j,j]+(j-3)*e[1,2,2]+(2-j)*e[1,3,3]+Rational((2*m-1)*(j-3),(2*m-3)*3)*e[2,3,4]
                               for j in range(4,2*m+1)])
manual_S1_perp_basis_segmented.append([e[1,j,k]-delta(k,j-1)*e[1,4,3]
                               for k in range(4,2*m) for j in range(k+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[2,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-3)*3)*e[2,3,4]
                               for k in range(5,2*m+1) for j in range(3,k)])
# # Something is wrong with the one below :(
manual_S1_perp_basis_segmented.append([e[2,j,j]+Rational(3-j,2)*e[1,2,1]+Rational(1-j,2)*e[2,3,3]
                               +Rational((2*m-2)*(j-1),2*m-1)*e[1,3,2]+Rational(-j*(2*m-3)-2*m-1,2*(2*m-1))*e[1,4,3]
                               for j in range(4,2*m+1)])
manual_S1_perp_basis_segmented.append([e[2,j,k]+delta(k,j-1)*(Rational(2*(2*m-2),(2*m-1))*e[1,4,2]-e[2,4,3])
                               for k in range(4,2*m) for j in range(k+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[3,4,4]-2*e[1,3,1]+3*e[2,3,2]+Rational(6*(2*m-2)-3*(2*m-3),2*m-1)*e[1,4,2]-3*e[2,4,3]])
manual_S1_perp_basis_segmented.append([e[3,j,4]-Rational(3*(2*m-3),2*m-1)*e[1,j,2]+delta(j,5)*e[2,3,1]
                              for j in range(5,2*m+1)])

manual_S1_perp_basis_segmented.append([e[3,j,k]+delta(k,j+1)*(Rational(j*(2*m-j),2*m-1)*(e[1,3,2]-e[1,4,3]))
                              for k in range(5,2*m+1) for j in range(4,k)])
manual_S1_perp_basis_segmented.append([e[3,j,j]+(2-j)*e[1,3,1]+(j-1)*e[2,3,2]+Rational(2*(2*m-2)*(j-1),2*m-1)*e[1,4,2]+(1-j)*e[2,4,3]
                              for j in range(5,2*m+1)])
manual_S1_perp_basis_segmented.append([e[3,j,k]+delta(k,j-1)*e[2,3,1]
                              for k in range(5,2*m) for j in range(k+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[4,j,4]+2*e[1,j,1]-3*e[2,j,2]+delta(j,5)*e[2,4,1]
                              for j in range(5,2*m+1)])

manual_S1_perp_basis_segmented.append([e[4,j,5]-Rational(4*(2*m-4),2*m-1)*e[1,j,2]+delta(j,5)*(4*e[2,4,2]-3*e[1,4,1])+delta(j,6)*e[2,4,1]
                              for j in range(5,2*m+1)])

manual_S1_perp_basis_segmented.append([e[4,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,4,2]
                              for k in range(6,2*m+1) for j in range(5,k)])
manual_S1_perp_basis_segmented.append([e[4,j,j]+(2-j)*e[1,4,1]+(j-1)*e[2,4,2]
                              for j in range(6,2*m+1)])
manual_S1_perp_basis_segmented.append([e[4,j,k]+delta(k,j-1)*e[2,4,1]
                               for k in range(6,2*m) for j in range(k+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,k]-delta(k,i-1)*e[2,j,1]
                              for i in range(5,2*m+1) for k in range(1,i) for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,i]+(i-2)*e[1,j,1]+(1-i)*e[2,j,2]+delta(i,j-1)*e[2,i,1]
                              for i in range(5,2*m+1) for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,i+1]-Rational(i*(2*m-i),2*m-1)*e[1,j,2]+delta(j,i+2)*e[2,i,1]+delta(j,i+1)*((1-i)*e[1,i,1]+i*e[2,i,2])
                              for i in range(5,2*m) for j in range(i+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,i,2]
                              for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(i+1,k)])
manual_S1_perp_basis_segmented.append([e[i,j,j]+(2-j)*e[1,i,1]+(j-1)*e[2,i,2]
                              for i in range(5,2*m-1) for j in range(i+2,2*m+1)])
manual_S1_perp_basis_segmented.append([e[i,j,k]+delta(k,j-1)*e[2,i,1]
                              for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(k+1,2*m+1)])

manual_S1_perp_basis_segmented.append([e[i,j,i]+(i-2)*e[1,j,1]+(1-i)*e[2,j,2]+delta(i,j-1)*e[2,i,1]
                                      for i in range(5,2*m+1) for j in range(i+1,2*m+1)])
manual_S1_perp_basis_segmented.append([e[i,j,i+1]-Rational(i*(2*m-i),2*m-1)*e[1,j,2]+delta(j,i+2)*e[2,i,1]
                                       +delta(j,i+1)*((1-i)*e[1,i,1]+i*(e[2,i,2]))
                                      for i in range(5,2*m) for j in range(i+1,2*m+1)])
manual_S1_perp_basis_segmented.append([e[i,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,i,2]
                                      for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(i+1,k)])
manual_S1_perp_basis_segmented.append([e[i,j,j]+(2-j)*e[1,i,1]+(j-1)*e[2,i,2]
                                      for i in range(5,2*m-1) for j in range(i+2,2*m+1)])
manual_S1_perp_basis_segmented.append([e[i,j,k]+delta(k,j-1)*e[2,i,1]
                                      for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(k+1,2*m+1)])

manual_S1_perp_basis=[]
for A in manual_S1_perp_basis_segmented:
    manual_S1_perp_basis+=A

In [26]:
# Testing manual_S1_perp_basis
set(S1_perp_basis)==set(manual_S1_perp_basis)

True

# Inner Product

In [27]:
inner_prod_dict={}
for i in range(1,2*m+1):
    for j in range(1,2*m+1):
        for k in range(1,2*m+1):
            if i==j:
                inner_prod_dict[e[i,j,k]]=0
            else:
                inner_prod_dict[e[i,j,k]]=Rational(factorial(k-1)*factorial(2*m-i)*factorial(2*m-j),
                                                   factorial(2*m-k)*factorial(i-1)*factorial(j-1))

In [28]:
## I haven't dealt with the antisymmetry issue here
def inner_prod(A,B):
    '''A,B: linear combinations of e[] elements
       returns: the inner product of A and B'''
    
    result=0
    if len(A.as_coefficients_dict())<=len(B.as_coefficients_dict()):
        first=A
        second=B
    else:
        first=B
        second=A
    for e_elt in first.as_coefficients_dict():
        if e_elt in second.as_coefficients_dict():
            result+=first.as_coefficients_dict()[e_elt]*second.as_coefficients_dict()[e_elt]*inner_prod_dict[e_elt]
        t=e_elt.indices
        rev_elt=e[t[1],t[0],t[2]]
        if rev_elt in second.as_coefficients_dict():
            result+=-first.as_coefficients_dict()[e_elt]*second.as_coefficients_dict()[rev_elt]*inner_prod_dict[e_elt]
    return result

We would like to compute $\text{im}(S^1)^\perp\cap (\omega\otimes V)^\perp=(\text{im}(S^1)+\omega\otimes V)^\perp$ where $\omega = \sum_{i=1}^{m}(-1)^i \varepsilon_i\wedge\varepsilon_{2m+1-i}$.

In [29]:
omega_V_basis=[sum([(-1)**i*e[i,2*m+1-i,k] for i in range(1,m+1)]) for k in range(1,2*m+1)]

omega_V_vecs=[coordinatize(C2_basis,A) for A in omega_V_basis]

S1_basis=[sum([e[i,b,b] for b in range(1,2*m+1)]).subs(antisymm_subs) for i in range(1,2*m+1)]
S1_basis+=[sum([e[i,b,b+1] for b in range(1,2*m)]).subs(antisymm_subs) for i in range(1,2*m+1)]
S1_basis+=[sum([(2*b-2*m-1)*e[i,b,b] for b in range(1,2*m+1)]).subs(antisymm_subs) for i in range(1,2*m+1)]
S1_basis+=[sum([(b-1)*(2*m+1-b)*e[i,b,b-1] for b in range(1,2*m+1)]).subs(antisymm_subs) for i in range(1,2*m+1)]

#S1_vecs=[coordinatize(C2_basis,A) for A in S1_basis]

In [30]:
# omega_V_perp_basis

omega_V_perp_basis=[e[i,j,k] for i in range(1,2*m+1) for j in range(i+1,2*m+1) for k in range(1,2*m+1) if j!=2*m+1-i]
omega_V_perp_basis+=[e[i,2*m+1-i,k]+e[i+1,2*m-i,k] for i in range(1,m) for k in range(1,2*m+1)]

omega_V_perp_vecs=[coordinatize(C2_basis,A) for A in omega_V_perp_basis]

In [31]:
# norm_cond_basis

norm_vecs=intersection(S1_perp_vecs,omega_V_perp_vecs)
norm_cond_basis=[coords_to_lin_comb(C2_basis,A) for A in norm_vecs]

In [32]:
def orthogonal_lists(list_A,list_B):
    '''list_A,list_B: lists of linear combinations of e[i,j,k]
       result: True if list_A is orthogonal to list_B, False otherwise'''
    for A in list_A:
        for B in list_B:
            if inner_prod(A,B)!=0:
#                 print(A)
#                 print(B)
#                 print()
                return False
    return True

In [33]:
orthogonal_lists(manual_S1_perp_basis,S1_basis)

True

In [34]:
# # This is for checking manual computations

# partial_norm_basis=[A for A in S1_perp_basis if orthogonal_lists([A],omega_V_basis)]

We'll call the basis for $\text{im}(S^1)^\perp\cap (\omega\otimes V)^\perp$  norm_basis

In [35]:
# To do: figure out what's going on here; finish

manual_norm_basis=[]

manual_norm_basis+=[e[i,j,1] for i in range(3,2*m+1) for j in range(i+1,2*m+1) if j!=2*m+1-i]
manual_norm_basis+=[e[i,j,2]-delta(i,3)*e[2,j,1] for i in range(3,2*m+1) for j in range(i+1,2*m+1) 
             if j!=2*m+1-i and (i!=3 or j!=2*m-1)]


manual_norm_basis+=[e[1,j,3] for j in range(5,2*m)]
manual_norm_basis+=[e[2,j,3]-Rational(2*(2*m-2),2*m-1)*e[1,j,2] for j in range(5,2*m-1)]
manual_norm_basis+=[e[3,j,3]+e[1,j,1]-2*e[2,j,2]+delta(j,4)*e[2,3,1] for j in range(4,2*m-2)]
manual_norm_basis+=[e[i,j,3]-delta(i,4)*e[2,j,1] for i in range(4,2*m+1) for j in range(i+1,2*m+1) 
             if j!=2*m+1-i and (i!=4 or j!=2*m-1)]


manual_norm_basis+=[e[1,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-2)*2)*e[1,2,3]
             for k in range(4,2*m+1) for j in range(2,k)]
manual_norm_basis+=[e[1,j,j]+(j-3)*e[1,2,2]+(2-j)*e[1,3,3]+Rational((2*m-1)*(j-3),(2*m-3)*3)*e[2,3,4] 
             for j in range(4,2*m)]
manual_norm_basis+=[e[1,j,k]-delta(k,j-1)*e[1,4,3] for k in range(4,2*m) for j in range(k+1,2*m)]
manual_norm_basis+=[e[2,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-3)*3)*e[2,3,4]
             for k in range(5,2*m+1) for j in range(3,k) if j<=2*m-2]
manual_norm_basis+=[e[2,j,j]+Rational(3-j,2)*e[1,2,1]+Rational(1-j,2)*e[2,3,3]+Rational((2*m-2)*(j-1),2*m-1)*e[1,3,2]
            +Rational(-j*(2*m-3)-2*m-1,2*(2*m-1))*e[1,4,3] for j in range(4,2*m+1) if j!=2*m-1]
manual_norm_basis+=[e[2,j,k]+delta(k,j-1)*(Rational(2*(2*m-2),2*m-1)*e[1,4,2]-e[2,4,3])
            for k in range(4,2*m) for j in range(k+1,2*m+1) if j!=2*m-1]
if m!=6:
    manual_norm_basis+=[e[3,4,4]-2*e[1,3,1]+3*e[2,3,2]+Rational(6*(2*m-2)-3*(2*m-3),2*m-1)*e[1,4,2]-3*e[2,4,3]]
manual_norm_basis+=[e[3,j,4]-Rational(3*(2*m-3),2*m-1)*e[1,j,2]+delta(j,5)*e[2,3,1]
             for j in range(5,2*m+1) if (j!=2*m-2 and j!=2*m)]
manual_norm_basis+=[e[3,j,k]+delta(k,j+1)*(Rational(j*(2*m-j),2*m-1)*(e[1,3,2]-e[1,4,3]))
             for k in range(5,2*m+1) for j in range(4,k) if j!=2*m-2]
manual_norm_basis+=[e[3,j,j]+(2-j)*e[1,3,1]+(j-1)*e[2,3,2]+Rational(2*(2*m-2)*(j-1),2*m-1)*e[1,4,2]+(1-j)*e[2,4,3]
            for j in range(5,2*m+1) if j !=2*m-2]
manual_norm_basis+=[e[3,j,k]+delta(k,j-1)*e[2,3,1] for k in range(5,2*m)
             for j in range(k+1,2*m+1) if j!=2*m-2]
manual_norm_basis+=[e[4,j,4]+2*e[1,j,1]-3*e[2,j,2]+delta(j,5)*e[2,4,1]
             for j in range(5,2*m+1) if j not in [2*m-3,2*m-1,2*m]]
manual_norm_basis+=[e[4,j,5]-Rational(4*(2*m-4),2*m-1)*e[1,j,2]+delta(j,5)*(4*e[2,4,2]-3*e[1,4,1])+delta(j,6)*e[2,4,1]
            for j in range(5,2*m+1) if j not in[2*m-3,2*m]]


manual_norm_basis+=[e[4,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,4,2]
             for k in range(6,2*m) for j in range(5,k) if j!=2*m-3]
manual_norm_basis+=[e[4,j,j]+(2-j)*e[1,4,1]+(j-1)*e[2,4,2] for j in range(6,2*m+1) if j!=2*m-3]
manual_norm_basis+=[e[4,j,k]+delta(k,j-1)*e[2,4,1] for k in range(6,2*m-1) for j in range(k+1, 2*m+1) if j!=2*m-3]
manual_norm_basis+=[e[i,j,k]-delta(k,i-1)*e[2,j,1]
             for i in range(5,1*m+1) for k in range(1,i) for j in range(i+1, 2*m+1) if j!=2*m+1-i]

In [36]:
# print(orthogonal_lists(manual_norm_basis,S1_basis))
# print(orthogonal_lists(manual_norm_basis,omega_V_basis))

## Basic Testing

In [37]:
# Test: Make sure determined_cijk agree with my computations
test_determined_cijk=set()
for i in range(1,3):
    for j in range(i+1,2*m+1):
        test_determined_cijk.add(c[i,j,1])

for i in range(1,3):
    for j in range(i+1,2*m+1):
        test_determined_cijk.add(c[i,j,2])
        
for t in [(1,2,3),(1,3,3),(1,4,3),(2,3,3),(2,4,3)]:
    test_determined_cijk.add(c[t[0],t[1],t[2]])
    
test_determined_cijk.add(c[(2,3,4)])
    
if test_determined_cijk==determined_cijk:
    print('determined_cijk agrees with my computations')

determined_cijk agrees with my computations


In [38]:
# Testing the manually generated basis
if set(manual_S1_perp_basis)==set(S1_perp_basis):
    print('passed manual_S1_perp_basis test')
else:
    print('failed manual_S1_perp_basis test')
    


# test1=set(manual_S1_perp_basis).difference(set(S1_perp_basis))
# test2=set(S1_perp_basis).difference(set(manual_S1_perp_basis))
# if test1!=set():
#     print(test1,'in manual_S1_perp_basis, but not S1_perp_basis')
# if test2!=set():
#     print(test2,'in S1_perp_basis, but not manual_S1_perp_basis')

passed manual_S1_perp_basis test


In [39]:
# Dimension test:
if 0==len(norm_cond_basis+S1_basis+omega_V_basis)-len(C2_basis):
    print('passed dimension test')
else:
    print('failed dimension test:')
    print('len(norm_cond_basis) = ',len(norm_cond_basis))
    print('len(S1_basis) =', len(S1_basis))
    print('len(omega_V_basis) =', len(omega_V_basis))

# Orthogonality test:
if orthogonal_lists(norm_cond_basis, S1_basis+omega_V_basis):
    print('passed orthogonality test')
else:
    print('failed orthogonality test')

passed dimension test
passed orthogonality test


In [40]:
# pickle norm_cond_basis 

file=open('eijk_norm_cond_m%d'%m,'wb') # 'wb' means 'write binary mode'
pickle.dump(norm_cond_basis,file)
file.close()